# Point Defects: StructureContainer Demo

Demonstrates `phase_diagram_workflows.structures.point_defects` -- building vacancy, substitution, and interstitial point defects on top of a `StructureContainer`, plus the interstitial site-finding and symmetry-unique sublattice discovery that feed it.

This covers **defect creation** only. Size-convergence and formation-energy calculation (the next layer, given a potential and these structures) are a separate, not-yet-built piece on top of this.

In [1]:
import sys
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ase.build import bulk

from phase_diagram_workflows.structures.point_defects import (
    add_pristine,
    create_interstitial,
    create_substitution,
    create_vacancy,
    create_vacancy_batch,
    discover_atomic_sublattices,
    discover_interstitial_sublattices,
    filter_by_generation,
    filter_by_operations_contains,
    get_structure_table,
    get_voronoi_interstitial_sites,
    validate_sublattice_coverage,
)

## 1. Pristine host

A `StructureContainer` starts empty; `add_pristine` adds a reference structure everything else is built on top of. A small FCC Al cell is enough to demonstrate every operation below quickly.

In [2]:
host = bulk("Al", cubic=True)
container = add_pristine(atoms=host, unique_id="Al_fcc")
print(container)
get_structure_table(container)

StructureContainer(1 structures: 1 pristine, 0 defects)


,structure,unique_id,is_pristine,stoichiometry,generation,pristine_structure_index,parent_index,operation,operations_short,events,metadata,creation_timestamp
0,"(Atom('Al', [np.float64(0.0), np.float64(0.0),...",Al_fcc,True,Al4,0,-1,-1,pristine,pristine,[],{},2026-09-16 21:43:39.909479


## 2. Vacancies

`create_vacancy` takes *either* `atom_ids` (explicit sites) *or* `n`/`seed` (reproducible random selection) -- exactly one of the two.

In [3]:
# Explicit: remove a specific atom
container = create_vacancy(container, atom_ids=[0])

# Random: remove 1 reproducibly-chosen atom from the (still-pristine) host again
container = create_vacancy(container, n=1, seed=0)

get_structure_table(container)[["unique_id", "is_pristine", "stoichiometry", "operation"]]

,unique_id,is_pristine,stoichiometry,operation
0,Al_fcc,True,Al4,pristine
1,defect_1,False,Al3,vacancy[0]
2,defect_2,False,Al3,vacancy[3]


## 3. Substitutions

Same explicit-vs-random split. Random mode additionally needs `from_element`, since there's no single atom index to read the starting species from.

In [4]:
container = create_substitution(container, to_element="Mg", atom_ids=[0])
container = create_substitution(container, to_element="Mg", n=1, seed=1, from_element="Al")

get_structure_table(container)[["unique_id", "stoichiometry", "operation"]].tail(2)

,unique_id,stoichiometry,operation
3,defect_3,Al3Mg1,substitution[Al->Mg]
4,defect_4,Al3Mg1,substitution[Al->Mg]


## 4. Batch creation

`create_vacancy_batch`/`create_substitution_batch`/`create_interstitial_batch` apply the same operation across several target structures at once. `separate_structures=True` (default) makes one new structure per site per target, rather than combining them.

In [5]:
pristine_idx = 0
container = create_vacancy_batch(
    container, target_indices=[pristine_idx], atom_ids=[1, 2], separate_structures=True
)
len(container.get_defect_structures())

6

## 5. Interstitials: site discovery + creation

Interstitials need candidate sites first. `get_voronoi_interstitial_sites` finds void centers geometrically (scipy Voronoi tessellation, no pymatgen dependency) -- `unique_sites` are one representative per cluster, `all_sites` are every deduplicated void.

In [6]:
unique_sites, all_sites = get_voronoi_interstitial_sites(host, r_min=0.5, cluster_tol=0.3)
print(f"{len(unique_sites)} unique interstitial sites, {len(all_sites)} total")

container = create_interstitial(container, sublattice=unique_sites, element="Mg", site_ids=[0])
get_structure_table(container)[["unique_id", "stoichiometry", "operation"]].tail(1)

12 unique interstitial sites, 12 total


,unique_id,stoichiometry,operation
7,defect_7,Al4Mg1,interstitial[Mg]


## 6. Symmetry-unique sublattices

For picking *one representative site per symmetry-distinct orbit* rather than enumerating every geometrically-equivalent atom/void -- this is what the production defect-formation-energy notebooks actually loop over (one vacancy/substitution/interstitial per orbit, not per atom).

In [7]:
atomic_orbits = discover_atomic_sublattices(host)
for orbit in atomic_orbits:
    print(f"  {orbit['label']}: species={orbit['species']}, multiplicity={orbit['multiplicity']}")
validate_sublattice_coverage(atomic_orbits, len(host))  # raises if orbits don't exactly cover the host

interstitial_orbits = discover_interstitial_sublattices(host, r_min=0.5, cluster_tol=0.3)
for orbit in interstitial_orbits:
    print(f"  {orbit['label']}: multiplicity={orbit['multiplicity']}")

  Al_4a: species=Al, multiplicity=4


  int_4b: multiplicity=4
  int_8c: multiplicity=8


## 7. Lineage and filtering

Defects can chain on top of each other via `parent_defect_index` -- here a vacancy, then a substitution built on top of that same vacancy structure, rather than on the pristine host.

In [8]:
container2 = add_pristine(atoms=bulk("Al", cubic=True))
container2 = create_vacancy(container2, atom_ids=[0])
vacancy_idx = len(container2) - 1
container2 = create_substitution(container2, to_element="Mg", atom_ids=[1], parent_defect_index=vacancy_idx)

chained = container2.get_structure(len(container2) - 1)
print(f"generation={chained['generation']}, events={[e['type'] for e in chained['events']]}")
print(f"operations_short={chained['operations_short']!r}")

generation=2, events=['vacancy', 'substitution']
operations_short='vacancy[0]|substitution[Al->Mg]'


In [9]:
# generation/operation-type filters over the whole container built above
print(f"{len(filter_by_generation(container, 1))} first-generation defects")
print(f"{len(filter_by_operations_contains(container, 'vacancy'))} structures with a vacancy in their history")

get_structure_table(container)[["unique_id", "is_pristine", "generation", "stoichiometry", "operation"]]

7 first-generation defects
4 structures with a vacancy in their history


,unique_id,is_pristine,generation,stoichiometry,operation
0,Al_fcc,True,0,Al4,pristine
1,defect_1,False,1,Al3,vacancy[0]
2,defect_2,False,1,Al3,vacancy[3]
3,defect_3,False,1,Al3Mg1,substitution[Al->Mg]
4,defect_4,False,1,Al3Mg1,substitution[Al->Mg]
5,defect_5,False,1,Al3,vacancy[1]
6,defect_6,False,1,Al3,vacancy[2]
7,defect_7,False,1,Al4Mg1,interstitial[Mg]
